# LangChain Structured Output

এই notebook-এ LangChain-এর **Structured Output** feature দেখানো হবে।

LLM সাধারণত free-form text return করে। কিন্তু real application-এ আমাদের দরকার হয় **নির্দিষ্ট format**-এ data — যাতে পরবর্তী code সেটা সরাসরি ব্যবহার করতে পারে।

`with_structured_output()` method দিয়ে LLM-কে বলা যায় — "এই schema অনুযায়ী output দাও।"

```
Raw Text  ──►  LLM + Schema  ──►  Typed Python Object
```

| Schema Type | Package | কখন ব্যবহার করবে |
|---|---|---|
| **Pydantic** (Basic) | `pydantic` | Validation দরকার হলে, simple flat structure |
| **Pydantic** (Nested) | `pydantic` | Complex structure, nested object বা list |
| **TypedDict** | `typing_extensions` | Lightweight, validation ছাড়া type hint |
| **DataClass** | `dataclasses` | Standard Python OOP, familiar syntax |

**সব example-এ একই `weather_db` ব্যবহার করা হয়েছে।**

### Step 1 — Environment Setup

**কী হচ্ছে:** `.env` file থেকে `ANTHROPIC_API_KEY` load করা হচ্ছে।

**কেন:** API key কখনো code-এ লেখা যাবে না। `python-dotenv` দিয়ে `.env` file থেকে পড়লে key টা safe থাকে।

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

ANTHROPIC_API_KEY: True


---
### Step 2 — Weather Tool এবং LLM তৈরি

**কী হচ্ছে:** আগের notebook-এর মতোই `get_weather` tool এবং `ChatAnthropic` LLM তৈরি করা হচ্ছে।

**Flow:**
```
get_weather(city)  →  raw text string
         ↓
LLM + Schema  →  Typed Python object
```

**কেন raw text থেকে structured output?** Real world-এ আমরা প্রায়ই API, document, বা tool থেকে unstructured text পাই — সেটাকে typed object-এ convert করার জন্যই structured output ব্যবহার করা হয়।

In [2]:
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic

@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city name."""
    weather_db = {
        "dhaka":      {"temperature": "34°C", "condition": "Sunny",         "humidity": "72%", "wind_speed": "10 km/h", "feels_like": "38°C"},
        "chittagong": {"temperature": "32°C", "condition": "Partly Cloudy", "humidity": "78%", "wind_speed": "14 km/h", "feels_like": "36°C"},
        "london":     {"temperature": "17°C", "condition": "Overcast",      "humidity": "85%", "wind_speed": "20 km/h", "feels_like": "15°C"},
    }
    key = city.lower().strip()
    if key not in weather_db:
        return f"No weather data available for '{city}'."
    w = weather_db[key]
    return (
        f"Weather in {city.title()}:\n"
        f"  Temperature : {w['temperature']} (feels like {w['feels_like']})\n"
        f"  Condition   : {w['condition']}\n"
        f"  Humidity    : {w['humidity']}\n"
        f"  Wind Speed  : {w['wind_speed']}"
    )

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

# Demo: raw tool output যেটাকে আমরা structure করবো
raw_dhaka = get_weather.invoke({"city": "Dhaka"})
print("Raw tool output (unstructured):")
print(raw_dhaka)

Raw tool output (unstructured):
Weather in Dhaka:
  Temperature : 34°C (feels like 38°C)
  Condition   : Sunny
  Humidity    : 72%
  Wind Speed  : 10 km/h


---
## Structured Output কীভাবে কাজ করে

**`with_structured_output(Schema)`** method LLM-কে বলে — "তুমি যা return করবে তা অবশ্যই এই schema মেনে চলবে।"

LangChain ভেতরে ভেতরে:
1. Schema টাকে JSON Schema-তে convert করে
2. সেটা LLM-কে tool/function হিসেবে দেয় (Anthropic-এর tool use feature)
3. LLM JSON output দেয়
4. LangChain সেই JSON কে Python object-এ parse করে return করে

```python
structured_llm = llm.with_structured_output(MySchema)
result = structured_llm.invoke("some prompt")
# result is now a MySchema instance, not a string!
print(type(result))  # <class 'MySchema'>
```

---
## 1. Pydantic Model — Basic (Flat Structure)

**কী হচ্ছে:** `BaseModel` দিয়ে একটা simple flat schema তৈরি করা হচ্ছে এবং সেই অনুযায়ী LLM-কে output দিতে বলা হচ্ছে।

**Pydantic-এর সুবিধা:**
- `Field(description=...)` দিয়ে LLM-কে hint দেওয়া যায় প্রতিটা field-এ কী লিখতে হবে
- Auto validation — wrong type দিলে error হয়
- `.model_dump()` দিয়ে সহজে dict-এ convert করা যায়

**কখন ব্যবহার করবে:** যখন data সহজ, flat এবং validation দরকার।

In [3]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage

# ── Schema Definition ──
class WeatherInfo(BaseModel):
    """Structured weather information for a city."""
    city:        str = Field(description="Name of the city")
    temperature: str = Field(description="Current temperature with unit (e.g. 34°C)")
    feels_like:  str = Field(description="Feels like temperature with unit")
    condition:   str = Field(description="Weather condition (e.g. Sunny, Cloudy)")
    humidity:    str = Field(description="Humidity percentage")
    wind_speed:  str = Field(description="Wind speed with unit")

# ── Bind schema to LLM ──
structured_llm = llm.with_structured_output(WeatherInfo)

# ── Get raw weather then parse into typed object ──
raw = get_weather.invoke({"city": "Dhaka"})

result: WeatherInfo = structured_llm.invoke(
    [HumanMessage(content=f"Extract structured weather data from this report:\n\n{raw}")]
)

print("Type   :", type(result))
print("Object :", result)
print()
print("── Individual fields (dot access) ──")
print("City        :", result.city)
print("Temperature :", result.temperature)
print("Feels Like  :", result.feels_like)
print("Condition   :", result.condition)
print("Humidity    :", result.humidity)
print("Wind Speed  :", result.wind_speed)
print()
print("── As dict (downstream use) ──")
print(result.model_dump())

Type   : <class '__main__.WeatherInfo'>
Object : city='Dhaka' temperature='34°C' feels_like='38°C' condition='Sunny' humidity='72%' wind_speed='10 km/h'

── Individual fields (dot access) ──
City        : Dhaka
Temperature : 34°C
Feels Like  : 38°C
Condition   : Sunny
Humidity    : 72%
Wind Speed  : 10 km/h

── As dict (downstream use) ──
{'city': 'Dhaka', 'temperature': '34°C', 'feels_like': '38°C', 'condition': 'Sunny', 'humidity': '72%', 'wind_speed': '10 km/h'}


---
## 2. Pydantic Model — Nested Structure

**কী হচ্ছে:** একটা model-এর ভেতরে আরেকটা model রাখা হচ্ছে — nested Pydantic objects। পাশাপাশি `list[str]` field দেখানো হচ্ছে।

**Nested Structure কেন দরকার:**
- Real data প্রায়ই hierarchical — যেমন `temperature` object-এর ভেতরে `actual` ও `feels_like` আলাদা থাকতে পারে
- `list[str]` field দিয়ে multiple item একসাথে রাখা যায়
- প্রতিটা nested model independently validate হয়

**এই example-এ:**
- `TemperatureDetails` → nested model, temperature breakdown
- `WeatherAlert` → nested model, alert level (`Literal`) + Bangla message
- `FullWeatherReport` → parent model যেটায় দুটো nested model এবং একটা `list[str]` আছে

**কখন ব্যবহার করবে:** Complex response যেখানে grouping বা hierarchy দরকার।

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage

# ── Nested Model 1: Temperature breakdown ──
class TemperatureDetails(BaseModel):
    """Temperature breakdown."""
    actual:     str = Field(description="Actual temperature reading (e.g. 17°C)")
    feels_like: str = Field(description="Feels like temperature (e.g. 15°C)")

# ── Nested Model 2: Alert ──
class WeatherAlert(BaseModel):
    """Weather alert for the city."""
    level:   Literal["low", "medium", "high"] = Field(
                 description="Alert severity — low (mild), medium (caution), high (extreme)")
    message: str = Field(description="Short alert message in Bengali")

# ── Parent Model: Full report with nested objects + list ──
class FullWeatherReport(BaseModel):
    """Complete weather report with nested structure."""
    city:        str                = Field(description="City name")
    temperature: TemperatureDetails = Field(description="Temperature details")
    condition:   str                = Field(description="Weather condition")
    humidity:    str                = Field(description="Humidity percentage")
    wind_speed:  str                = Field(description="Wind speed")
    alert:       WeatherAlert       = Field(description="Weather alert")
    tips:        list[str]          = Field(description="Exactly 3 practical weather tips in Bengali")

# ── Bind schema to LLM ──
structured_llm = llm.with_structured_output(FullWeatherReport)

raw = get_weather.invoke({"city": "London"})

result: FullWeatherReport = structured_llm.invoke(
    [HumanMessage(content=f"Extract a full structured weather report from this:\n\n{raw}")]
)

print("Type   :", type(result).__name__)

print("\n── City ──")
print(result.city)

print("\n── Temperature (nested TemperatureDetails object) ──")
print("  Actual     :", result.temperature.actual)
print("  Feels Like :", result.temperature.feels_like)

print("\n── Alert (nested WeatherAlert object) ──")
print("  Level   :", result.alert.level)
print("  Message :", result.alert.message)

print("\n── Tips (list[str]) ──")
for i, tip in enumerate(result.tips, 1):
    print(f"  {i}. {tip}")

print("\n── Full dict (model_dump, nested too) ──")
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))

Type   : FullWeatherReport

── City ──
London

── Temperature (nested TemperatureDetails object) ──
  Actual     : 17°C
  Feels Like : 15°C

── Alert (nested WeatherAlert object) ──
  Level   : low
  Message : কোনো সতর্কতা নেই

── Tips (list[str]) ──
  1. হালকা জ্যাকেট পরে বাইরে যান কারণ বাতাস ঠান্ডা অনুভব হবে
  2. উচ্চ আর্দ্রতার কারণে ছাতা নিয়ে যাওয়া সুবিধাজনক হতে পারে
  3. বাতাসের গতি মাঝারি তাই হালকা পোশাক পরুন

── Full dict (model_dump, nested too) ──
{
  "city": "London",
  "temperature": {
    "actual": "17°C",
    "feels_like": "15°C"
  },
  "condition": "Overcast",
  "humidity": "85%",
  "wind_speed": "20 km/h",
  "alert": {
    "level": "low",
    "message": "কোনো সতর্কতা নেই"
  },
  "tips": [
    "হালকা জ্যাকেট পরে বাইরে যান কারণ বাতাস ঠান্ডা অনুভব হবে",
    "উচ্চ আর্দ্রতার কারণে ছাতা নিয়ে যাওয়া সুবিধাজনক হতে পারে",
    "বাতাসের গতি মাঝারি তাই হালকা পোশাক পরুন"
  ]
}


: 

---
## 3. TypedDict

**কী হচ্ছে:** `TypedDict` দিয়ে schema define করা হচ্ছে। এটা Pydantic-এর মতোই কাজ করে কিন্তু অনেক lightweight এবং output হয় plain `dict`।

**Pydantic থেকে পার্থক্য:**

| বিষয় | Pydantic | TypedDict |
|---|---|---|
| Runtime validation | ✅ হয় | ❌ হয় না |
| Dependency | `pydantic` দরকার | শুধু `typing_extensions` |
| Return type | Pydantic object | Plain `dict` |
| Field description | `Field(description=...)` | `Annotated[type, "description"]` |

**`Annotated` কেন:** TypedDict-এ `Field()` নেই, তাই `Annotated[type, "description string"]` দিয়ে LLM-কে field description পাঠানো হয়।

**কখন ব্যবহার করবে:** যখন শুধু type hint দরকার, validation না হলেও চলবে, এবং output সরাসরি `dict` হিসেবে ব্যবহার করতে হবে।

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage

# ── Schema Definition (TypedDict) ──
class WeatherSummary(TypedDict):
    """Summarized weather info with heat classification."""
    city:        Annotated[str, "Name of the city"]
    temperature: Annotated[str, "Current temperature with unit"]
    condition:   Annotated[str, "Weather condition description"]
    heat_level:  Annotated[
                     Literal["cold", "comfortable", "hot", "very_hot"],
                     "Heat classification: cold (<15°C) / comfortable (15-25°C) / hot (25-35°C) / very_hot (>35°C)"
                 ]
    suggestion:  Annotated[str, "One-sentence suggestion in Bengali about what to wear or do"]

# ── Bind schema to LLM ──
structured_llm = llm.with_structured_output(WeatherSummary)

raw = get_weather.invoke({"city": "Chittagong"})

result: WeatherSummary = structured_llm.invoke(
    [HumanMessage(content=f"Summarize and classify this weather data:\n\n{raw}")]
)

print("Type   :", type(result))   # dict — class instance নয়!
print("Result :", result)
print()
print("── Field access (dict style) ──")
print("City        :", result["city"])
print("Temperature :", result["temperature"])
print("Heat Level  :", result["heat_level"])
print("Suggestion  :", result["suggestion"])

# TypedDict result is a plain dict — downstream logic সহজ
print()
print("── Downstream logic ──")
if result["heat_level"] in ("hot", "very_hot"):
    print("⚠ গরম আবহাওয়া সতর্কতা!")
else:
    print("✓ আবহাওয়া ঠিকঠাক।")

---
## 4. Python DataClass

**কী হচ্ছে:** Python standard library-র `@dataclass` decorator দিয়ে schema define করা হচ্ছে। কোনো third-party package দরকার নেই।

**DataClass-এর বৈশিষ্ট্য:**
- Extra dependency নেই — শুধু `from dataclasses import dataclass, field`
- Dot notation দিয়ে field access: `result.city`
- `field(default=...)` দিয়ে default value দেওয়া যায়
- `dataclasses.asdict()` দিয়ে dict-এ convert করা যায়

**তিনটা schema-র তুলনা:**

| বিষয় | Pydantic | TypedDict | DataClass |
|---|---|---|---|
| Runtime validation | ✅ | ❌ | ❌ |
| Extra dependency | `pydantic` | `typing_extensions` | শুধু stdlib |
| Return type | object | `dict` | object |
| Field access | `obj.field` | `obj["field"]` | `obj.field` |
| JSON export | `.model_dump()` | সরাসরি dict | `dataclasses.asdict()` |

**কখন ব্যবহার করবে:** Pure Python project যেখানে Pydantic install নেই, বা simple data container দরকার।

> **Note:** `with_structured_output` দিয়ে dataclass support করে `langchain-core >= 0.3` থেকে।

In [ ]:
import dataclasses
from dataclasses import dataclass, field
from langchain_core.messages import HumanMessage

# ── Schema Definition (DataClass) ──
@dataclass
class WeatherData:
    """Structured weather data with Bangla advice."""
    city:        str
    temperature: str
    feels_like:  str
    condition:   str
    humidity:    str
    wind_speed:  str
    advice:      str = field(
                     default="",
                     metadata={"description": "One practical advice sentence in Bengali for this weather"}
                 )

# ── Bind schema to LLM ──
structured_llm = llm.with_structured_output(WeatherData)

raw = get_weather.invoke({"city": "Dhaka"})

result: WeatherData = structured_llm.invoke(
    [HumanMessage(
        content=(
            f"Extract weather data from this report and write one practical advice in Bengali:\n\n{raw}"
        )
    )]
)

print("Type   :", type(result).__name__)   # WeatherData — dataclass instance
print("Object :", result)
print()
print("── Field access (dot notation) ──")
print("City        :", result.city)
print("Temperature :", result.temperature)
print("Feels Like  :", result.feels_like)
print("Condition   :", result.condition)
print("Advice      :", result.advice)
print()
print("── As dict (dataclasses.asdict) ──")
print(dataclasses.asdict(result))
print()
print("── Is it a dataclass? ──")
print(dataclasses.is_dataclass(result))   # True

---
## সারসংক্ষেপ

| Schema Type | Define করার উপায় | Return Type | Validation | Serialization |
|---|---|---|---|---|
| **Pydantic Basic** | `class X(BaseModel)` | Pydantic instance | ✅ | `.model_dump()` |
| **Pydantic Nested** | Model inside model + `list[T]` | Pydantic instance | ✅ | `.model_dump()` (nested-ও কাজ করে) |
| **TypedDict** | `class X(TypedDict)` | `dict` | ❌ | সরাসরি dict |
| **DataClass** | `@dataclass class X` | dataclass instance | ❌ | `dataclasses.asdict()` |

**সব ক্ষেত্রে একই interface:**
```python
structured_llm = llm.with_structured_output(YourSchema)
result = structured_llm.invoke([HumanMessage(content="...")])
```

**কোনটা বেছে নেবে?**
- Production API / strict validation → **Pydantic Basic**
- Complex hierarchical data → **Pydantic Nested**
- Simple, lightweight, dict output → **TypedDict**
- No extra deps, standard Python → **DataClass**